# GTFS Data Processing Workflow

This notebook provides a complete, step-by-step workflow for cleaning, merging, validating, and processing GTFS data for intercity transit feeds. The goal is to prepare the data for mapping and analysis in GIS tools like ArcGIS Pro or QGIS.

## Notes
- Paths, filenames, and outputs are parameterized in **Configuration** (no user-specific paths).
- External dependencies are listed; install them before running.
- An interactive route-filtering step is preserved (optional; can be disabled).
- Destructive steps (overwrites/deletes) are clearly labeled.

## Prerequisites
- Python 3.x with libraries: pandas, gtfs_kit, folium, and others as needed.
- `gtfstidy` installed and available on PATH (see Step 6).
- All GTFS feeds should be placed in a single root directory, with each agency's feed in its own subfolder containing the standard GTFS files (agency.txt, routes.txt, etc.).
- Update the **Configuration** section below to point to your GTFS feeds directory and output locations.

## Configuration

In [ ]:
import shutil
import tempfile
from pathlib import Path
from collections import defaultdict
import re
from datetime import datetime
import pandas as pd
import gtfs_kit as gk
from typing import Dict, List, Set, Tuple
import folium
from folium import FeatureGroup, LayerControl, Map, PolyLine, CircleMarker, Popup
from folium.plugins import MarkerCluster, BeautifyIcon
from itertools import cycle
import math
import os, csv, json
import subprocess
import zipfile

## 1. Download GTFS Feeds

Download or collect GTFS feeds from transit agencies or sources like Transitland, Mobility Database, etc. Place all unzipped GTFS folders into the directory configured by `ROOT_DIR` (by default `DATA_ROOT/gtfs_feeds/`). Ensure each subfolder contains the standard GTFS files (e.g., `agency.txt`, `routes.txt`, `trips.txt`, `stop_times.txt`, `calendar.txt`, etc.). Once you have all feeds in `ROOT_DIR`, proceed to the next steps.

In [ ]:
# Configuration (edit these before running)
PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "data"  # place GTFS feeds here
ROOT_DIR = DATA_ROOT / "gtfs_feeds"  # each agency in its own subfolder
RUN_TAG = "2026-01-20"  # update per run (e.g., 2026-01-20)

# Output directories
UNIQUE_FEEDS_DIR = DATA_ROOT / "Unique_Feeds"
MERGED_FEED_DIR = DATA_ROOT / "Merged_Feed"
MERGED_TIDY_FEED_DIR = Path(str(MERGED_FEED_DIR) + "_tidy")
OUTPUT_DIR = PROJECT_ROOT / "outputs"

# Output files
WEEKLY_TRIPS_CSV = OUTPUT_DIR / f"weekly_trips_per_shape_{RUN_TAG}.csv"
GEOJSON_OUTPUT = OUTPUT_DIR / f"Merged_Agencies_{RUN_TAG}.geojson"
MAP_OUTPUT = OUTPUT_DIR / f"Intercity_bus_map_{RUN_TAG}.html"

# Interactive route filtering (Step 4)
INTERACTIVE_ROUTE_FILTERING = True  # keep True for manual input
ROUTE_IDS_TO_REMOVE = []  # used when INTERACTIVE_ROUTE_FILTERING is False

# Create the directories if they don't exist
for _dir in [UNIQUE_FEEDS_DIR, MERGED_FEED_DIR, OUTPUT_DIR]:
    _dir.mkdir(parents=True, exist_ok=True)

print("Configuration loaded:")
print(f"  ROOT_DIR = {ROOT_DIR}")
print(f"  OUTPUT_DIR = {OUTPUT_DIR}")

## 2. Split Combined Feeds

Some agencies share one GTFS feed that covers multiple bus operators. To make the data easier to clean and work with, split these into separate feeds per agency.

This code reads agency.txt, filters related files (routes, trips, stop_times, stops, shapes, calendar), and creates individual agency folders in `UNIQUE_FEEDS_DIR`.

In [ ]:
# Code for splitting combined GTFS feeds

def clean_folder_name(name):
    # Remove or replace characters not allowed in folder names
    return ''.join(c for c in name if c.isalnum() or c in (' ', '_', '-')).rstrip()

def split_gtfs_by_agency(gtfs_folder, output_base_folder):
    # Load the GTFS feed using gtfs_kit
    feed = gk.read_feed(gtfs_folder, dist_units='km')
    
    # Get all agency_ids
    agencies = feed.agency
    agency_ids = agencies['agency_id'].unique() if 'agency_id' in agencies.columns else [None]
    
    for agency_id in agency_ids:
        # Get agency name
        if 'agency_id' in agencies.columns and 'agency_name' in agencies.columns:
            agency_name = agencies.loc[agencies['agency_id'] == agency_id, 'agency_name'].values[0]
        elif 'agency_name' in agencies.columns:
            agency_name = agencies['agency_name'].values[0]
        else:
            agency_name = str(agency_id) or 'default'
        
        out_dir = os.path.join(output_base_folder, clean_folder_name(agency_name))
        os.makedirs(out_dir, exist_ok=True)
        
        print(f"Creating feed for agency: {agency_name}")
        
        # Create a new feed object for this agency
        agency_feed = gk.Feed(dist_units='km')
        
        # Filter agency
        if 'agency_id' in agencies.columns:
            agency_feed.agency = agencies[agencies['agency_id'] == agency_id].copy()
        else:
            agency_feed.agency = agencies.copy()
        
        # Filter routes by agency_id
        if feed.routes is not None and 'agency_id' in feed.routes.columns:
            agency_feed.routes = feed.routes[feed.routes['agency_id'] == agency_id].copy()
        else:
            agency_feed.routes = feed.routes.copy() if feed.routes is not None else pd.DataFrame()
        
        if agency_feed.routes.empty:
            print(f"No routes found for {agency_name}, skipping.")
            continue
        
        route_ids = set(agency_feed.routes['route_id'])
        
        # Filter trips by route_id
        if feed.trips is not None:
            agency_feed.trips = feed.trips[feed.trips['route_id'].isin(route_ids)].copy()
        else:
            agency_feed.trips = pd.DataFrame()
        
        if agency_feed.trips.empty:
            print(f"No trips found for {agency_name}, skipping.")
            continue
        
        trip_ids = set(agency_feed.trips['trip_id'])
        service_ids = set(agency_feed.trips['service_id'])
        shape_ids = set(agency_feed.trips['shape_id'].dropna()) if 'shape_id' in agency_feed.trips.columns else set()
        
        # Filter stop_times by trip_id
        if feed.stop_times is not None:
            agency_feed.stop_times = feed.stop_times[feed.stop_times['trip_id'].isin(trip_ids)].copy()
        else:
            agency_feed.stop_times = pd.DataFrame()
        
        stop_ids = set(agency_feed.stop_times['stop_id']) if not agency_feed.stop_times.empty else set()
        
        # Filter stops by stop_id
        # NOTE: this WILL drop stops not referenced by stop_times for this agency.
        if feed.stops is not None and stop_ids:
            # Snapshot BEFORE for reporting
            _stops_before = feed.stops.copy()

            agency_feed.stops = feed.stops[feed.stops['stop_id'].isin(stop_ids)].copy()

            # Report/export dropped stops (from the *combined* feed perspective)
            try:
                before_ids = set(_stops_before['stop_id'].astype(str))
                after_ids = set(agency_feed.stops['stop_id'].astype(str))
                dropped_ids = sorted(before_ids - after_ids)
                if dropped_ids:
                    dropped_df = _stops_before[_stops_before['stop_id'].astype(str).isin(dropped_ids)].copy()
                    out_csv = Path(out_dir) / "dropped_stops_after_split.csv"
                    dropped_df.to_csv(out_csv, index=False, encoding='utf-8')
                    print(f"  → Dropped {len(dropped_ids)} stops during split for {agency_name} (exported: {out_csv})")
                    print(f"    Examples: {', '.join(dropped_ids[:10])}")
                else:
                    print(f"  → No stops dropped during split for {agency_name}")
            except Exception as _exc:
                print(f"  ⚠ Could not compute dropped stops during split for {agency_name}: {_exc}")
        else:
            agency_feed.stops = feed.stops.copy() if feed.stops is not None else pd.DataFrame()
        
        # Filter shapes by shape_id
        if feed.shapes is not None and shape_ids:
            agency_feed.shapes = feed.shapes[feed.shapes['shape_id'].isin(shape_ids)].copy()
        else:
            agency_feed.shapes = pd.DataFrame()
        
        # Filter calendar by service_id
        if feed.calendar is not None:
            agency_feed.calendar = feed.calendar[feed.calendar['service_id'].isin(service_ids)].copy()
        else:
            agency_feed.calendar = pd.DataFrame()
        
        # Filter calendar_dates by service_id
        if feed.calendar_dates is not None:
            agency_feed.calendar_dates = feed.calendar_dates[feed.calendar_dates['service_id'].isin(service_ids)].copy()
        else:
            agency_feed.calendar_dates = pd.DataFrame()
        
        # Filter fare_rules and fare_attributes if they exist
        if feed.fare_rules is not None and 'route_id' in feed.fare_rules.columns:
            agency_feed.fare_rules = feed.fare_rules[feed.fare_rules['route_id'].isin(route_ids)].copy()
            if not agency_feed.fare_rules.empty and feed.fare_attributes is not None:
                fare_ids = set(agency_feed.fare_rules['fare_id'])
                agency_feed.fare_attributes = feed.fare_attributes[feed.fare_attributes['fare_id'].isin(fare_ids)].copy()
        
        # Filter transfers by stop_ids
        if feed.transfers is not None and stop_ids:
            agency_feed.transfers = feed.transfers[
                feed.transfers['from_stop_id'].isin(stop_ids) &
                feed.transfers['to_stop_id'].isin(stop_ids)
            ].copy()
        
        # Copy other optional tables if they exist
        if feed.frequencies is not None:
            agency_feed.frequencies = feed.frequencies[feed.frequencies['trip_id'].isin(trip_ids)].copy() if 'trip_id' in feed.frequencies.columns else pd.DataFrame()
        
        # Write the feed to the output directory
        agency_feed.to_file(str(out_dir))
        print(f"✅ Created feed for {agency_name} in {out_dir}")
    
    print(f"Done! Created {len(agency_ids)} folders in {output_base_folder}.")

# Auto-detect and split any multi-agency GTFS feeds under ROOT_DIR
# A feed is considered "multi-agency" if agency.txt has >1 agency_id (or >1 row when agency_id missing)

required_tables = ["agency.txt", "routes.txt", "trips.txt", "stop_times.txt", "stops.txt"]

candidates = []
for feed_dir in ROOT_DIR.iterdir():
    if not feed_dir.is_dir():
        continue
    if not all((feed_dir / tbl).exists() for tbl in required_tables):
        continue
    try:
        agency_df = pd.read_csv(feed_dir / "agency.txt")
        if "agency_id" in agency_df.columns:
            agency_ids = agency_df["agency_id"].dropna().unique()
        else:
            agency_ids = agency_df.index  # fall back to row count
        if len(agency_ids) > 1:
            candidates.append(feed_dir)
    except Exception as exc:
        print(f"Skipping {feed_dir} (could not inspect agencies: {exc})")

if not candidates:
    print("No multi-agency feeds found under ROOT_DIR to split.")
else:
    print(f"Found {len(candidates)} multi-agency feed(s):")
    for feed_dir in candidates:
        print(f"  - {feed_dir}")
    for feed_dir in candidates:
        target_base = UNIQUE_FEEDS_DIR / feed_dir.name
        target_base.mkdir(parents=True, exist_ok=True)
        print(f"\nSplitting {feed_dir} into per-agency feeds under {target_base}...")
        split_gtfs_by_agency(str(feed_dir), str(target_base))
    print("Done splitting all detected multi-agency feeds.")


## 3. Validate GTFS Feeds

Validate each GTFS feed in the root directory. Load required files, check for missing fields, unique IDs, referential integrity, time/date formats, coordinates, and stop sequences. Use loops to process each subfolder and report validation results.

In [ ]:
orphaned_per_agency = {}
missing_per_agency = {}
unused_per_agency = {}
identical_per_agency = {}
single_day_per_agency = {}
unused_stops_per_agency = {}

# GTFS file definitions
required_files = [
    'agency.txt',
    'stops.txt',
    'routes.txt',
    'trips.txt',
    'stop_times.txt',
    'calendar.txt'
]

optional_files = [
    'calendar_dates.txt',
    'shapes.txt'
]

required_fields = {
    'agency': ['agency_id', 'agency_name'],
    'stops': ['stop_id', 'stop_name', 'stop_lat', 'stop_lon'],
    'routes': ['route_id', 'agency_id'],
    'trips': ['route_id', 'service_id', 'trip_id'],
    'stop_times': ['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence'],
    'calendar': ['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']
}

def check_required_files(gtfs_data, required_files):
    """Check if all required GTFS files are present."""
    missing = [f for f in required_files if f.replace('.txt', '') not in gtfs_data]
    if missing:
        print(f"Missing required files: {missing}")
    else:
        print("All required files are present.")
    return missing

def check_unique_ids(df, id_column, file_name):
    """Check for unique IDs in a dataframe."""
    duplicates = df[df.duplicated(id_column, keep=False)]
    if not duplicates.empty:
        print(f"Duplicate {id_column} in {file_name}:")
        print(duplicates[id_column].unique())
        return False
    else:
        print(f"All {id_column} are unique in {file_name}.")
        return True

def check_referential_integrity(primary_df, primary_key, foreign_df, foreign_key, primary_name, foreign_name):
    """Check if foreign keys exist in primary table."""
    missing = foreign_df[~foreign_df[foreign_key].isin(primary_df[primary_key])]
    if not missing.empty:
        print(f"Missing {primary_key} in {primary_name} referenced by {foreign_name}:")
        print(missing[foreign_key].unique())
        return False
    else:
        print(f"All {foreign_key} in {primary_name} exist in {foreign_name}.")
        return True

def validate_time_format(time_str):
    """Validate GTFS time format (HH:MM:SS, can be >24:00:00 for next day)."""
    if pd.isna(time_str):
        return True
    pattern = re.compile(r'^(\d+):([0-5]\d):([0-5]\d)$')
    return bool(pattern.match(str(time_str)))

def validate_date_format(date_str):
    """Validate GTFS date format (YYYYMMDD)."""
    if pd.isna(date_str):
        return True
    try:
        datetime.strptime(str(date_str), '%Y%m%d')
        return True
    except ValueError:
        return False

def validate_lat_lon(lat, lon):
    """Validate latitude and longitude ranges."""
    try:
        lat = float(lat)
        lon = float(lon)
        return -90 <= lat <= 90 and -180 <= lon <= 180
    except (ValueError, TypeError):
        return False

def check_stop_sequences(stop_times_df):
    """Check if stop sequences are consecutive for each trip."""
    errors = []
    for trip_id, group in stop_times_df.groupby('trip_id'):
        sequences = sorted(group['stop_sequence'].unique())
        if not sequences:
            continue
        min_seq = min(sequences)
        expected = list(range(min_seq, min_seq + len(sequences)))
        if sequences != expected:
            errors.append((trip_id, sequences, expected))
    if errors:
        print("Stop sequence errors:")
        for trip_id, actual, expected in errors:
            print(f"Trip {trip_id}: expected {expected}, got {actual}")
    else:
        print("All stop sequences are consecutive.")
    return errors
# Validate each GTFS feed
subdirs = [d for d in os.listdir(ROOT_DIR) if os.path.isdir(os.path.join(ROOT_DIR, d))]
for subdir in subdirs:
    feed_dir = os.path.join(ROOT_DIR, subdir)
    print(f"\n{'='*50}")
    print(f"Validating GTFS feed: {subdir}")
    print(f"{'='*50}")
    
    gtfs_data = {}
    for file in required_files + optional_files:
        file_path = os.path.join(feed_dir, file)
        if os.path.exists(file_path):
            try:
                gtfs_data[file.replace('.txt', '')] = pd.read_csv(file_path)
                print(f"✓ Loaded {file}")
            except Exception as e:
                print(f"✗ Error loading {file}: {e}")
        else:
            if file in required_files:
                print(f"✗ Missing required file: {file}")
            else:
                print(f"⚠ Missing optional file: {file}")
    
    # Check for missing required files
    missing_required = [f for f in required_files if f.replace('.txt', '') not in gtfs_data]
    if missing_required:
        print(f"❌ Skipping further checks for {subdir} due to missing required files: {missing_required}")
        continue
    
    print(f"✓ All required files present for {subdir}")
    
    # Check required fields
    all_fields_ok = True
    for file_key, req_fields in required_fields.items():
        if file_key in gtfs_data:
            missing_fields = [f for f in req_fields if f not in gtfs_data[file_key].columns]
            if missing_fields:
                print(f"✗ Missing required fields in {file_key}.txt: {missing_fields}")
                all_fields_ok = False
            else:
                print(f"✓ All required fields present in {file_key}.txt")
    
    # Special check for routes: needs route_short_name or route_long_name
    if 'routes' in gtfs_data:
        if 'route_short_name' not in gtfs_data['routes'].columns and 'route_long_name' not in gtfs_data['routes'].columns:
            print("✗ Missing route name (route_short_name or route_long_name) in routes.txt")
            all_fields_ok = False
        else:
            print("✓ Route name field present in routes.txt")
    
    if not all_fields_ok:
        print(f"❌ Skipping further validations for {subdir} due to missing fields")
        continue
    
    print(f"✓ All required fields present for {subdir}")
    
    # Proceed with data validations
    print(f"\n--- Data Validations for {subdir} ---")
    
    # Check unique IDs
    unique_ok = True
    if not check_unique_ids(gtfs_data['routes'], 'route_id', 'routes.txt'):
        unique_ok = False
    if not check_unique_ids(gtfs_data['trips'], 'trip_id', 'trips.txt'):
        unique_ok = False
    if not check_unique_ids(gtfs_data['stops'], 'stop_id', 'stops.txt'):
        unique_ok = False
    
    # Check referential integrity
    ref_ok = True
    if not check_referential_integrity(gtfs_data['agency'], 'agency_id', gtfs_data['routes'], 'agency_id', 'agency.txt', 'routes.txt'):
        ref_ok = False
    if not check_referential_integrity(gtfs_data['routes'], 'route_id', gtfs_data['trips'], 'route_id', 'routes.txt', 'trips.txt'):
        ref_ok = False
    if not check_referential_integrity(gtfs_data['stops'], 'stop_id', gtfs_data['stop_times'], 'stop_id', 'stops.txt', 'stop_times.txt'):
        ref_ok = False
    if not check_referential_integrity(gtfs_data['trips'], 'trip_id', gtfs_data['stop_times'], 'trip_id', 'trips.txt', 'stop_times.txt'):
        ref_ok = False
    if not check_referential_integrity(gtfs_data['calendar'], 'service_id', gtfs_data['trips'], 'service_id', 'calendar.txt', 'trips.txt'):
        ref_ok = False
    
    # Check stop sequences
    seq_errors = check_stop_sequences(gtfs_data['stop_times'])
    seq_ok = len(seq_errors) == 0
    
    # Validate data formats
    format_ok = True
    
    print("\nValidating time formats in stop_times...")
    invalid_times = gtfs_data['stop_times'][~gtfs_data['stop_times']['arrival_time'].apply(validate_time_format)]
    if not invalid_times.empty:
        print(f"✗ Invalid arrival times found: {len(invalid_times)} entries")
        format_ok = False
    
    invalid_times_dep = gtfs_data['stop_times'][~gtfs_data['stop_times']['departure_time'].apply(validate_time_format)]
    if not invalid_times_dep.empty:
        print(f"✗ Invalid departure times found: {len(invalid_times_dep)} entries")
        format_ok = False
    
    print("Validating date formats in calendar...")
    invalid_dates = gtfs_data['calendar'][~gtfs_data['calendar']['start_date'].apply(validate_date_format) | ~gtfs_data['calendar']['end_date'].apply(validate_date_format)]
    if not invalid_dates.empty:
        print(f"✗ Invalid dates found: {len(invalid_dates)} entries")
        format_ok = False
    
    print("Validating lat/lon in stops...")
    invalid_coords = gtfs_data['stops'][~gtfs_data['stops'].apply(lambda row: validate_lat_lon(row['stop_lat'], row['stop_lon']), axis=1)]
    if not invalid_coords.empty:
        print(f"✗ Invalid coordinates found: {len(invalid_coords)} entries")
        format_ok = False
    
    # Additional validations
    print("\n--- Additional Validations ---")
    
    additional_ok = True
    
    # 1. Trip_ids in stop_times not in trips
    if 'stop_times' in gtfs_data and 'trips' in gtfs_data:
        stop_times_trip_ids = set(gtfs_data['stop_times']['trip_id'])
        trips_trip_ids = set(gtfs_data['trips']['trip_id'])
        orphaned_trip_ids = stop_times_trip_ids - trips_trip_ids
        if orphaned_trip_ids:
            print(f"✗ Found {len(orphaned_trip_ids)} trip_ids in stop_times not in trips: {sorted(list(orphaned_trip_ids)[:5])}...")
            additional_ok = False
        else:
            print("✓ All trip_ids in stop_times exist in trips")
        
        orphaned_per_agency[subdir] = orphaned_trip_ids
    
    # 2. Trips with service_id not in calendar
    if 'trips' in gtfs_data and 'calendar' in gtfs_data:
        calendar_service_ids = set(gtfs_data['calendar']['service_id'])
        trips_service_ids = set(gtfs_data['trips']['service_id'])
        missing_service_ids = trips_service_ids - calendar_service_ids
        if missing_service_ids:
            print(f"✗ Found {len(missing_service_ids)} service_ids in trips not in calendar: {sorted(list(missing_service_ids)[:5])}...")
            additional_ok = False
        else:
            print("✓ All service_ids in trips exist in calendar")
        
        missing_per_agency[subdir] = missing_service_ids
    
    # 3. Unused shape_ids
    if 'shapes' in gtfs_data and 'trips' in gtfs_data:
        shapes_shape_ids = set(gtfs_data['shapes']['shape_id']) if 'shape_id' in gtfs_data['shapes'].columns else set()
        trips_shape_ids = set(gtfs_data['trips']['shape_id'].dropna()) if 'shape_id' in gtfs_data['trips'].columns else set()
        unused_shape_ids = shapes_shape_ids - trips_shape_ids
        if unused_shape_ids:
            print(f"✗ Found {len(unused_shape_ids)} unused shape_ids in shapes: {sorted(list(unused_shape_ids)[:5])}...")
            additional_ok = False
        else:
            print("✓ All shape_ids in shapes are used in trips")
        
        unused_per_agency[subdir] = unused_shape_ids
    
    # 4. Identical shape_ids
    if 'shapes' in gtfs_data:
        shape_sequences = {}
        for shape_id, group in gtfs_data['shapes'].groupby('shape_id'):
            group = group.sort_values('shape_pt_sequence')
            sequence = tuple(zip(group['shape_pt_lat'], group['shape_pt_lon']))
            shape_sequences[shape_id] = sequence
        sequence_groups = defaultdict(list)
        for shape_id, seq in shape_sequences.items():
            sequence_groups[seq].append(shape_id)
        identical_groups = {seq: ids for seq, ids in sequence_groups.items() if len(ids) > 1}
        if identical_groups:
            total_identical = sum(len(ids) for ids in identical_groups.values())
            print(f"✗ Found {len(identical_groups)} groups of identical shapes ({total_identical} total shapes)")
            for seq, ids in list(identical_groups.items())[:3]:
                print(f"  Identical: {', '.join(sorted(str(id) for id in ids))}")
            additional_ok = False
        else:
            print("✓ No identical shapes found")
        
        identical_per_agency[subdir] = identical_groups
    
    # 5. Service_ids where start_date == end_date
    if 'calendar' in gtfs_data:
        single_day_services = gtfs_data['calendar'][gtfs_data['calendar']['start_date'] == gtfs_data['calendar']['end_date']]
        if not single_day_services.empty:
            print(f"✗ Found {len(single_day_services)} service_ids where start_date == end_date: {single_day_services['service_id'].tolist()[:5]}...")
            additional_ok = False
        else:
            print("✓ No service_ids with start_date == end_date")
        
        single_day_per_agency[subdir] = set(single_day_services['service_id']) if not single_day_services.empty else set()
    
    # 6. Unused stops
    if 'stops' in gtfs_data and 'stop_times' in gtfs_data:
        stops_stop_ids = set(gtfs_data['stops']['stop_id'])
        stop_times_stop_ids = set(gtfs_data['stop_times']['stop_id'])
        unused_stop_ids = stops_stop_ids - stop_times_stop_ids
        if unused_stop_ids:
            print(f"✗ Found {len(unused_stop_ids)} unused stop_ids in stops: {sorted(list(unused_stop_ids)[:5])}...")
            additional_ok = False
        else:
            print("✓ All stop_ids in stops are used in stop_times")
        
        unused_stops_per_agency[subdir] = unused_stop_ids
    
    # Summary for this feed
    overall_ok = unique_ok and ref_ok and seq_ok and format_ok and additional_ok
    if overall_ok:
        print(f"\n✅ {subdir}: All validations passed!")
    else:
        print(f"\n❌ {subdir}: Some validations failed. Check details above.")
    
print(f"\n{'='*50}")
print("Validation complete for all GTFS feeds.")
print(f"{'='*50}")

## 4. Filter Intercity Routes

Review and identify routes from `routes.txt` files that provide only intercity services. Keep the intercity routes and remove the others. This script loads `routes.txt` to extract route IDs to retain, then filters other GTFS files by relationships.

The total route distance is calculated using the Haversine formula, which computes the great-circle distance between consecutive latitude/longitude points along the route's shape (from `shapes.txt`). Distances are summed for all points in sequence, and it assumes straight-line paths over Earth's surface. If a route has multiple shapes, distances are aggregated. This provides an accurate approximation but doesn't account for actual road paths.

> Note: This step is optional when all routes in a feed represent intercity services. Route filtering is only necessary for feeds that include both intercity and local routes, and it can be skipped to reduce processing time when not needed.

In [ ]:
# Filter intercity routes for each agency
# Note: This supports interactive input OR a predefined list via configuration

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    return R * c

# Collect all routes from all agencies
all_routes = []
agency_feeds = {}

for agency_folder in os.listdir(ROOT_DIR):
    agency_path = ROOT_DIR / agency_folder
    if not agency_path.is_dir():
        continue
    
    try:
        # Load feed with gtfs_kit
        feed = gk.read_feed(str(agency_path), dist_units='km')
        agency_feeds[agency_folder] = feed
        
        if feed.routes is not None:
            for idx, row in feed.routes.iterrows():
                route_name = row.get('route_long_name', row.get('route_short_name', 'No name'))
                
                # Calculate total distance for the route
                shape_ids = feed.trips[feed.trips['route_id'] == row['route_id']]['shape_id'].dropna().unique()
                total_dist = 0.0
                for shape_id in shape_ids:
                    if feed.shapes is not None:
                        shape_points = feed.shapes[feed.shapes['shape_id'] == shape_id].sort_values('shape_pt_sequence')
                        if not shape_points.empty:
                            for i in range(1, len(shape_points)):
                                p1_lat = shape_points.iloc[i-1]['shape_pt_lat']
                                p1_lon = shape_points.iloc[i-1]['shape_pt_lon']
                                p2_lat = shape_points.iloc[i]['shape_pt_lat']
                                p2_lon = shape_points.iloc[i]['shape_pt_lon']
                                total_dist += haversine(p1_lat, p1_lon, p2_lat, p2_lon)
                
                all_routes.append(f"{agency_folder}: {row['route_id']} - {route_name} ({total_dist * 0.621371:.2f} mi)")
    except Exception as e:
        print(f"Error loading {agency_folder}: {e}")

print("All routes across agencies:")
for route in all_routes:
    print(f"- {route}")

# Interactive input OR predefined list
if INTERACTIVE_ROUTE_FILTERING:
    to_remove = input("\nEnter route_ids to remove, separated by commas (or leave blank for none): ").strip()
    if to_remove:
        remove_list = [r.strip() for r in to_remove.split(',')]
        print(f"Routes to remove: {remove_list}")
    else:
        remove_list = []
        print("No routes to remove.")
else:
    remove_list = ROUTE_IDS_TO_REMOVE or []
    print(f"Using predefined route_ids to remove: {remove_list}")

# Now filter for each agency
for agency_folder, feed in agency_feeds.items():
    agency_path = ROOT_DIR / agency_folder
    
    print(f"\n{'='*50}")
    print(f"Filtering routes for {agency_folder}")
    print(f"{'='*50}")
    
    if feed.routes is None or feed.routes.empty:
        print(f"No routes found for {agency_folder}, skipping.")
        continue
    
    # Get all route_ids for this agency
    all_route_ids = set(feed.routes['route_id'])
    kept_routes = all_route_ids - set(remove_list)
    
    if len(kept_routes) == len(all_route_ids):
        print(f"No routes removed for {agency_folder}")
        continue
    
    print(f"Keeping {len(kept_routes)} out of {len(all_route_ids)} routes")
    
    # Filter routes
    feed.routes = feed.routes[feed.routes['route_id'].isin(kept_routes)].copy()
    
    # Filter trips by route_id
    if feed.trips is not None:
        feed.trips = feed.trips[feed.trips['route_id'].isin(kept_routes)].copy()
        kept_trip_ids = set(feed.trips['trip_id'])
        kept_service_ids = set(feed.trips['service_id'])
        kept_shape_ids = set(feed.trips['shape_id'].dropna()) if 'shape_id' in feed.trips.columns else set()
    else:
        print(f"No trips found for {agency_folder}")
        continue
    
    # Filter stop_times by trip_id
    if feed.stop_times is not None:
        feed.stop_times = feed.stop_times[feed.stop_times['trip_id'].isin(kept_trip_ids)].copy()
        kept_stop_ids = set(feed.stop_times['stop_id'])
    else:
        kept_stop_ids = set()
    
    # Filter stops by stop_id
    if feed.stops is not None and kept_stop_ids:
        _stops_before = feed.stops.copy()
        feed.stops = feed.stops[feed.stops['stop_id'].isin(kept_stop_ids)].copy()

        # Report/export dropped stops for this agency
        try:
            before_ids = set(_stops_before['stop_id'].astype(str))
            after_ids = set(feed.stops['stop_id'].astype(str))
            dropped_ids = sorted(before_ids - after_ids)
            if dropped_ids:
                dropped_df = _stops_before[_stops_before['stop_id'].astype(str).isin(dropped_ids)].copy()
                out_csv = Path(agency_path) / "dropped_stops_after_route_filter.csv"
                dropped_df.to_csv(out_csv, index=False, encoding='utf-8')
                print(f"  → Dropped {len(dropped_ids)} stops after route filtering (exported: {out_csv})")
                print(f"    Examples: {', '.join(dropped_ids[:10])}")
            else:
                print("  → No stops dropped after route filtering")
        except Exception as _exc:
            print(f"  ⚠ Could not compute dropped stops after route filtering: {_exc}")
    
    # Filter calendar by service_id
    if feed.calendar is not None:
        feed.calendar = feed.calendar[feed.calendar['service_id'].isin(kept_service_ids)].copy()
    
    # Filter calendar_dates by service_id
    if feed.calendar_dates is not None:
        feed.calendar_dates = feed.calendar_dates[feed.calendar_dates['service_id'].isin(kept_service_ids)].copy()
    
    # Filter fare_rules and fare_attributes
    if feed.fare_rules is not None and 'route_id' in feed.fare_rules.columns:
        feed.fare_rules = feed.fare_rules[feed.fare_rules['route_id'].isin(kept_routes)].copy()
        if not feed.fare_rules.empty and feed.fare_attributes is not None:
            kept_fare_ids = set(feed.fare_rules['fare_id'])
            feed.fare_attributes = feed.fare_attributes[feed.fare_attributes['fare_id'].isin(kept_fare_ids)].copy()
    
    # Filter shapes by shape_id
    if feed.shapes is not None and kept_shape_ids:
        feed.shapes = feed.shapes[feed.shapes['shape_id'].isin(kept_shape_ids)].copy()
    
    # Filter transfers by stop_ids
    if feed.transfers is not None and kept_stop_ids:
        feed.transfers = feed.transfers[
            feed.transfers['from_stop_id'].isin(kept_stop_ids) &
            feed.transfers['to_stop_id'].isin(kept_stop_ids)
        ].copy()
    
    # Filter frequencies by trip_id
    if feed.frequencies is not None:
        feed.frequencies = feed.frequencies[feed.frequencies['trip_id'].isin(kept_trip_ids)].copy()
    
    # Write the filtered feed back
    try:
        feed.to_file(str(agency_path))
        print(f"✅ Successfully filtered and saved {agency_folder}")
    except Exception as e:
        print(f"❌ Error saving {agency_folder}: {e}")

print(f"\n{'='*50}")
print("Filtering complete for all agencies.")
print(f"{'='*50}")

## 5. Consolidate Feeds

Merge all cleaned GTFS feeds into one combined feed.

- The notebook iterates agency folders under `ROOT_DIR`, loads each feed with `gtfs_kit`, and concatenates GTFS tables with `pd.concat()`.
- The merged output is written to `MERGED_FEED_DIR`.
- Note: IDs are not prefixed/remapped in this step; `gtfstidy -i` in Step 6 can renumber IDs later.

In [ ]:
input_folder = str(ROOT_DIR)

# Collect all feeds
feeds = []
for agency_folder in os.listdir(input_folder):
    agency_path = os.path.join(input_folder, agency_folder)
    if not os.path.isdir(agency_path):
        continue
    
    try:
        feed = gk.read_feed(agency_path, dist_units='km')
        feeds.append(feed)
        print(f"✅ Loaded: {agency_folder}")
    except Exception as e:
        print(f"❌ Error loading {agency_folder}: {e}")

# Append all feeds
if feeds:
    merged_feed = feeds[0]
    for feed in feeds[1:]:
        # Manually append feeds by concatenating DataFrames
        for attr in ['agency', 'stops', 'routes', 'trips', 'stop_times', 'calendar', 'calendar_dates', 'shapes']:
            if hasattr(feed, attr) and getattr(feed, attr) is not None:
                if hasattr(merged_feed, attr) and getattr(merged_feed, attr) is not None:
                    setattr(merged_feed, attr, pd.concat([getattr(merged_feed, attr), getattr(feed, attr)], ignore_index=True))
                else:
                    setattr(merged_feed, attr, getattr(feed, attr).copy())
    
    # Write to directory
    merged_feed.to_file(str(MERGED_FEED_DIR))
    print(f"✅ Merged GTFS saved to: {MERGED_FEED_DIR}")
else:
    print("❌ No feeds to merge.")

## 6. Tidy and Optimize with gtfstidy

This section prepares the merged GTFS feed for final output. The process is split into three distinct stages to ensure data integrity and maximum optimization:

### 1. Preparation (Python)
Before running any external tools, Python is used to strictly format the CSV files. This step is critical to prevent `gtfstidy` from crashing due to bad data.
- **Fixes**: Standardizes `agency.txt` (timezone/URL), removes orphaned `parent_station` in `stops.txt`, fixes hex colors in `routes.txt`, and removes orphaned `shape_id` references in `trips.txt`.
- **Zipping**: Packages the cleaned text files into a valid GTFS ZIP archive.

### 2. Modification (gtfstidy)
Run `gtfstidy` with **modification flags** (lowercase) to fix validation issues and normalize the data structure without aggressively deleting content yet.
- `--fix`: Automatically fixes common validation errors.
- `-e` (drop invalid on parse), `-m` (remeasure), `-s` (simplify), `-c` (compact), `-i` (renumber IDs).

### 3. Optimization (gtfstidy)
Run `gtfstidy` with **optimization flags** (uppercase / aggressive options) to clean and deduplicate the feed.
- **Used by default in this notebook**: `--delete-orphans`, `--drop-errs`, `-O`, `-S`, `-R`, `-C`, `-T`, `--red-stops-fuzzy`, `--snap-stops`, `-W`.
- **Optional**: `-P` (redundant stop removal) is intentionally commented out in the code because it can remove many non-redundant stops. Enable it only after validating stop counts/locations.

### 6.1 Preparation Phase
This step uses Python to clean and format the CSV files before any external tools are run.
*   **Goal**: Prevent `gtfstidy` crashes by fixing critical data issues (orphaned IDs, missing required fields).
*   **Key Actions**: Fix `agency.txt` timezone/URL, remove orphaned `parent_station` in `stops.txt`, fix hex colors in `routes.txt`.
*   **Output**: A clean `Merged_Feed.zip` ready for processing.

In [ ]:
print("🔧 1. PREPARATION: Preprocessing and Zipping...")

import csv


def read_gtfs_csv(file_path, encoding='utf-8-sig'):
    """Read a GTFS .txt as strings (preserve IDs + blanks)."""
    return pd.read_csv(
        file_path,
        dtype=str,
        encoding=encoding,
        low_memory=False,
        keep_default_na=False,
        na_filter=False,
    )


def _normalize_blank_series(s):
    # Keep blanks as blanks; normalize common literal NA strings.
    s = s.astype(str)
    s = s.replace({'nan': '', 'NaN': '', 'NaT': '', 'None': '', '<NA>': ''})
    return s.str.strip()


def _normalize_intish_series(s, *, col_name=None):
    """Normalize integer-ish columns without filling missing values.

    - '' stays ''
    - '1.0' -> '1'
    - invalid non-empty values are cleared to '' (to avoid tool crashes)
    """
    s0 = _normalize_blank_series(s)
    # Convert blanks to NA for numeric parsing
    num = pd.to_numeric(s0.where(s0 != '', pd.NA), errors='coerce')

    is_int = num.notna() & ((num % 1) == 0)
    out = pd.Series('', index=s0.index, dtype=object)
    out[is_int] = num[is_int].astype('Int64').astype(str)

    invalid = (s0 != '') & (~is_int)
    if col_name and invalid.any():
        print(f"  → Clearing {invalid.sum()} invalid integer values in {col_name}")

    return out


# Define helper function to clean and write CSV files
def clean_and_write_csv(file_path, df, encoding='utf-8'):
    """Write CSV in a GTFS-friendly, ID-preserving format."""
    df = df.copy()

    # Clean column names
    df.columns = df.columns.str.strip()

    # Strip whitespace and normalize literal NA strings
    for col in df.columns:
        df[col] = _normalize_blank_series(df[col])

    # Normalize integer-ish fields, but DO NOT fill missing values
    intish_fields = [
        'location_type',
        'wheelchair_boarding',
        'route_type',
        'direction_id',
        'pickup_type',
        'drop_off_type',
        'bikes_allowed',
        'wheelchair_accessible',
        'route_sort_order',
        'stop_sequence',
        'shape_pt_sequence',
        'payment_method',
        'transfers',
    ]
    for field in intish_fields:
        if field in df.columns:
            df[field] = _normalize_intish_series(df[field], col_name=field)

    # Final fill (should be minimal with read_gtfs_csv)
    df = df.fillna('')

    # Write CSV with minimal quoting
    with open(file_path, 'w', encoding=encoding, newline='') as f:
        df.to_csv(
            f,
            index=False,
            quoting=csv.QUOTE_MINIMAL,
            lineterminator='\n',
        )

# 0. Remove single-day services (start_date == end_date) from calendar/trips/stop_times
# This keeps the feed consistent by removing dependent trips + stop_times too.
def remove_single_day_services(feed_dir):
    calendar_path = feed_dir / 'calendar.txt'
    if not calendar_path.exists():
        return

    calendar_df = read_gtfs_csv(calendar_path)
    if 'service_id' not in calendar_df.columns:
        print(f"  → calendar.txt missing service_id; skipping single-day removal")
        return

    if 'start_date' not in calendar_df.columns or 'end_date' not in calendar_df.columns:
        print("  → calendar.txt missing start_date/end_date; skipping single-day removal")
        return

    start = calendar_df['start_date'].astype(str).replace('nan', '').replace('NaT', '').str.strip()
    end = calendar_df['end_date'].astype(str).replace('nan', '').replace('NaT', '').str.strip()
    single_day_mask = (start != '') & (end != '') & (start == end)
    single_day_service_ids = set(calendar_df.loc[single_day_mask, 'service_id'].astype(str))

    if not single_day_service_ids:
        print("✓ No single-day services found (start_date == end_date)")
        return

    print(f"  → Removing {len(single_day_service_ids)} single-day service_ids (start_date == end_date)")

    # calendar.txt
    before = len(calendar_df)
    calendar_df = calendar_df[~calendar_df['service_id'].astype(str).isin(single_day_service_ids)]
    print(f"    - calendar.txt: removed {before - len(calendar_df)} rows")
    clean_and_write_csv(calendar_path, calendar_df)

    # calendar_dates.txt (if present)
    calendar_dates_path = feed_dir / 'calendar_dates.txt'
    if calendar_dates_path.exists():
        cal_dates_df = read_gtfs_csv(calendar_dates_path)
        if 'service_id' in cal_dates_df.columns:
            before = len(cal_dates_df)
            cal_dates_df = cal_dates_df[~cal_dates_df['service_id'].astype(str).isin(single_day_service_ids)]
            print(f"    - calendar_dates.txt: removed {before - len(cal_dates_df)} rows")
            clean_and_write_csv(calendar_dates_path, cal_dates_df)

    # trips.txt + stop_times.txt
    trips_path = feed_dir / 'trips.txt'
    if trips_path.exists():
        trips_df = read_gtfs_csv(trips_path)
        if 'service_id' in trips_df.columns:
            before = len(trips_df)
            trips_df = trips_df[~trips_df['service_id'].astype(str).isin(single_day_service_ids)]
            print(f"    - trips.txt: removed {before - len(trips_df)} rows")
            clean_and_write_csv(trips_path, trips_df)

            if 'trip_id' in trips_df.columns:
                kept_trip_ids = set(trips_df['trip_id'].astype(str))
                stop_times_path = feed_dir / 'stop_times.txt'
                if stop_times_path.exists():
                    st_df = read_gtfs_csv(stop_times_path)
                    if 'trip_id' in st_df.columns:
                        before = len(st_df)
                        st_df = st_df[st_df['trip_id'].astype(str).isin(kept_trip_ids)]
                        print(f"    - stop_times.txt: removed {before - len(st_df)} rows")
                        clean_and_write_csv(stop_times_path, st_df)

# Run single-day removal early so later fixes operate on filtered data
remove_single_day_services(MERGED_FEED_DIR)

# 1. Fix agency.txt - CRITICAL: standardize timezone and ensure required fields
agency_path = MERGED_FEED_DIR / 'agency.txt'
if agency_path.exists():
    agency_df = read_gtfs_csv(agency_path)
    
    # Fix timezone (Required to be consistent)
    agency_df['agency_timezone'] = 'America/New_York'
    
    # Fix missing agency_url (Required field)
    if 'agency_url' not in agency_df.columns:
        print("  → Adding missing required field: agency_url")
        agency_df['agency_url'] = 'https://example.com'
    
    # Ensure no empty URLs and fix format
    agency_df['agency_url'] = agency_df['agency_url'].fillna('https://example.com').replace('', 'https://example.com')
    # Prepend https:// if missing
    needs_http = ~agency_df['agency_url'].astype(str).str.match(r'^https?://', na=False)
    if needs_http.any():
        print(f"  → Fixing {needs_http.sum()} malformed agency_url entries")
        agency_df.loc[needs_http, 'agency_url'] = 'https://' + agency_df.loc[needs_http, 'agency_url'].astype(str)

    # Fix agency_email (Optional field, but must be valid if present)
    if 'agency_email' in agency_df.columns:
        # Remove invalid emails (no '@')
        invalid_emails = (agency_df['agency_email'].notna()) & (agency_df['agency_email'] != '') & (~agency_df['agency_email'].astype(str).str.contains('@'))
        if invalid_emails.any():
            print(f"  → Clearing {invalid_emails.sum()} invalid agency_email entries (e.g. '{agency_df.loc[invalid_emails, 'agency_email'].iloc[0]}')")
            agency_df.loc[invalid_emails, 'agency_email'] = ''

    clean_and_write_csv(agency_path, agency_df)
    print(f"✓ Fixed agency.txt - standardized timezone, ensured agency_url, cleaned emails ({len(agency_df)} agencies)")

# 2. Fix stops.txt - remove orphaned parent_station references (gtfstidy can't parse if parent doesn't exist)
stops_path = MERGED_FEED_DIR / 'stops.txt'
if stops_path.exists():
    stops_df = read_gtfs_csv(stops_path)
    
    if 'parent_station' in stops_df.columns:
        stops_df['parent_station'] = _normalize_blank_series(stops_df['parent_station']).replace('nan.0', '')
    
    # Remove orphaned parent_station references (critical parsing issue)
    # NOTE: IDs are opaque strings; we preserve them and only strip whitespace.
    if 'parent_station' in stops_df.columns and 'stop_id' in stops_df.columns:
        stops_df['stop_id'] = _normalize_blank_series(stops_df['stop_id'])
        stops_df['parent_station'] = _normalize_blank_series(stops_df['parent_station']).replace('nan.0', '')

        valid_station_ids = set(stops_df['stop_id'])
        orphan_mask = (stops_df['parent_station'] != '') & (~stops_df['parent_station'].isin(valid_station_ids))
        orphaned_count = int(orphan_mask.sum())
        if orphaned_count > 0:
            print(f"  → Removing {orphaned_count} orphaned parent_station references")
            stops_df.loc[orphan_mask, 'parent_station'] = ''
    
    clean_and_write_csv(stops_path, stops_df)
    print(f"✓ Fixed stops.txt - removed orphaned references ({len(stops_df)} stops)")

# 3. Fix shapes.txt - remove shape_dist_traveled and fix sequences
shapes_path = MERGED_FEED_DIR / 'shapes.txt'
if shapes_path.exists():
    shapes_df = read_gtfs_csv(shapes_path)
    
    # Drop shape_dist_traveled (let gtfstidy recalculate)
    if 'shape_dist_traveled' in shapes_df.columns:
        print("  → Dropping shape_dist_traveled from shapes.txt")
        shapes_df.drop(columns=['shape_dist_traveled'], inplace=True)
    
    # Fix duplicate/bad shape_pt_sequence
    if 'shape_pt_sequence' in shapes_df.columns:
        # Sort numerically (sequences are read as strings)
        shapes_df['_shape_pt_sequence_num'] = pd.to_numeric(shapes_df['shape_pt_sequence'], errors='coerce')
        shapes_df.sort_values(['shape_id', '_shape_pt_sequence_num'], inplace=True)
        shapes_df.drop(columns=['_shape_pt_sequence_num'], inplace=True)

        # Re-index to ensure 0, 1, 2... strictly increasing per shape
        shapes_df['shape_pt_sequence'] = shapes_df.groupby('shape_id').cumcount()
        print("  → Re-indexed shape_pt_sequence to ensure strict ordering")

    clean_and_write_csv(shapes_path, shapes_df)
    print(f"✓ Fixed shapes.txt ({len(shapes_df)} points)")

# 4. Fix stop_times.txt - remove shape_dist_traveled and fix sequences
stop_times_path = MERGED_FEED_DIR / 'stop_times.txt'
if stop_times_path.exists():
    st_df = read_gtfs_csv(stop_times_path)
    
    # Drop shape_dist_traveled (let gtfstidy recalculate)
    if 'shape_dist_traveled' in st_df.columns:
        print("  → Dropping shape_dist_traveled from stop_times.txt")
        st_df.drop(columns=['shape_dist_traveled'], inplace=True)

    # Fix duplicate stop_sequence
    if 'stop_sequence' in st_df.columns:
        # Normalize integer-ish values but keep blanks blank
        st_df['stop_sequence'] = _normalize_intish_series(st_df['stop_sequence'], col_name='stop_sequence')

        # Sort numerically (sequences are read as strings)
        st_df['_stop_sequence_num'] = pd.to_numeric(st_df['stop_sequence'], errors='coerce')
        st_df.sort_values(['trip_id', '_stop_sequence_num'], inplace=True)

        # Check for duplicates or collisions
        if st_df.duplicated(subset=['trip_id', 'stop_sequence']).any():
             print("  → Re-indexing duplicate stop_sequence entries")
             st_df['stop_sequence'] = st_df.groupby('trip_id').cumcount() + 1

        st_df.drop(columns=['_stop_sequence_num'], inplace=True)

    clean_and_write_csv(stop_times_path, st_df)
    print(f"✓ Fixed stop_times.txt ({len(st_df)} rows)")

# 5. Fix other files (just ensure proper formatting)
for file_name in ['routes.txt', 'trips.txt', 'calendar.txt']:
    file_path = MERGED_FEED_DIR / file_name
    if file_path.exists():
        df = read_gtfs_csv(file_path)
        
        # Clean column names (strip whitespace)
        df.columns = df.columns.str.strip()

        # Specific fix for routes.txt
        if file_name == 'routes.txt':
             # Fix route_color and route_text_color (Must be 6-char hex)
             for color_col in ['route_color', 'route_text_color']:
                 if color_col in df.columns:
                     # Convert to string, strip whitespace/nan
                     df[color_col] = df[color_col].astype(str).replace('nan', '').str.strip()
                     # Remove '#'
                     df[color_col] = df[color_col].str.replace('#', '', regex=False)
                     
                     # Pad with leading zeros if valid hex but short (e.g. "9966" -> "009966")
                     # Filter for non-empty values
                     mask = df[color_col] != ''
                     if mask.any():
                         # Simple zfill for alphanumeric values
                         df.loc[mask, color_col] = df.loc[mask, color_col].apply(lambda x: x.zfill(6) if len(x) < 6 else x[:6])
                         # Ensure only valid hex chars remain, otherwise clear it
                         # (gtfstidy will fail on non-hex)
                         is_hex = df[color_col].str.match(r'^[0-9A-Fa-f]{6}$')
                         invalid_count = (~is_hex & mask).sum()
                         if invalid_count > 0:
                             print(f"  → Clearing {invalid_count} invalid {color_col} values")
                             df.loc[~is_hex, color_col] = ''
                         else:
                             print(f"  → Normalized {color_col} to 6-char hex")

             # route_sort_order is optional; keep blanks blank (but normalize int-ish values)
             if 'route_sort_order' in df.columns:
                 df['route_sort_order'] = _normalize_intish_series(df['route_sort_order'], col_name='route_sort_order')
             
             # Fix route_sort_color (likely a typo or invalid field causing validation errors)
             # Check case-insensitive
             cols_to_drop = [c for c in df.columns if c.lower() == 'route_sort_color']
             if cols_to_drop:
                 print(f"  → Dropping invalid/non-standard field(s) {cols_to_drop} from routes.txt")
                 df.drop(columns=cols_to_drop, inplace=True)
             
             # Double check it's gone
             if any(c.lower() == 'route_sort_color' for c in df.columns):
                 print("  WARNING: route_sort_color still present!")

        # Specific fix for trips.txt
        if file_name == 'trips.txt':
             # Remove orphaned shape_ids (trips referencing shapes that don't exist)
             if 'shape_id' in df.columns:
                 shapes_path = MERGED_FEED_DIR / 'shapes.txt'
                 if shapes_path.exists():
                     # Load shapes to get valid IDs (read as strings)
                     shapes_df = read_gtfs_csv(shapes_path)

                     valid_shapes = set(_normalize_blank_series(shapes_df['shape_id'])) if 'shape_id' in shapes_df.columns else set()

                     # Normalize trip shape_ids for checking
                     trip_shape_ids = _normalize_blank_series(df['shape_id'])
                     
                     # Find orphans: non-empty shape_ids that are not in valid_shapes
                     mask = (trip_shape_ids != '') & (~trip_shape_ids.isin(valid_shapes))
                     
                     if mask.any():
                         print(f"  → Removing {mask.sum()} orphaned shape_id references from trips.txt")
                         df.loc[mask, 'shape_id'] = ''

        clean_and_write_csv(file_path, df)
        print(f"✓ Fixed {file_name} formatting ({len(df)} rows)")

print("\n📦 Creating ZIP file for gtfstidy...")

# Remove existing zip if it exists to ensure fresh creation
zip_path = str(MERGED_FEED_DIR) + ".zip"
if os.path.exists(zip_path):
    os.remove(zip_path)

# Create ZIP with proper structure
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=9) as zipf:
    for txt_file in MERGED_FEED_DIR.glob('*.txt'):
        zipf.write(txt_file, arcname=txt_file.name)

print(f"✓ ZIP created: {zip_path}")

### 6.2 Modification Phase
This step runs `gtfstidy` with **lowercase flags** to normalize the feed and fix validation errors. It produces an intermediate `_modified.zip` file.
*   **Goal**: Ensure the feed is valid and consistent before aggressive optimization.
*   **Key Flags**: `--fix`, `-m`, `-s`, `-c`, `-i`.

In [ ]:
print("\n🚀 2. MODIFICATION: Running gtfstidy (Lowercase Flags)...")
# Intermediate output for the modified feed
intermediate_zip = str(MERGED_FEED_DIR) + "_modified.zip"
zip_path = str(MERGED_FEED_DIR) + ".zip" # Re-define zip_path as it might be lost in new cell context if not persisted (though variables persist in Jupyter, it's safer to be explicit or assume persistence)

# Run gtfstidy with flags to normalize the feed
# -e: drop invalid entries on parse
# -m: remeasure shapes
# -s: simplify/minimize shapes
# -c: minimize services (calendar/calendar_dates)
# -i: minimize/renumber IDs
# --fix: automatically fix common validation issues

# Remove existing output to avoid stale results
if os.path.exists(intermediate_zip):
    os.remove(intermediate_zip)

cmd = [
    'gtfstidy',
    '--fix',
    '-e',
    '-m',
    '-s',
    '-c',
    '-i',
    '-o',
    intermediate_zip,
    zip_path,
]

print(f"Command: {subprocess.list2cmdline(cmd)}\n")

try:
    result_mod = subprocess.run(cmd, capture_output=True, text=True)
except FileNotFoundError:
    raise RuntimeError("gtfstidy not found on PATH. Install it or add it to PATH, then re-run this cell.")

if result_mod.stdout:
    print("Output:")
    print(result_mod.stdout)
if result_mod.stderr:
    print("Errors/Warnings:")
    print(result_mod.stderr)

if result_mod.returncode == 0:
    print(f"\n✅ SUCCESS! GTFS feed modified (Stage 1).")
    
    print("\n📊 DETAILED MODIFICATION REPORT")
    print("==============================")

    try:
        import zipfile
        
        def get_df(path, filename):
            """Load a GTFS table as strings, preserving blanks."""
            p = Path(path)
            if p.is_file() and p.suffix == '.zip':
                with zipfile.ZipFile(p) as z:
                    if filename in z.namelist():
                        return pd.read_csv(
                            z.open(filename),
                            dtype=str,
                            keep_default_na=False,
                            na_filter=False,
                        )
            elif p.is_dir():
                f = p / filename
                if f.exists():
                    return pd.read_csv(
                        f,
                        dtype=str,
                        keep_default_na=False,
                        na_filter=False,
                    )
            return None

        # Load Dataframes
        files_to_load = ['stops.txt', 'routes.txt', 'trips.txt', 'shapes.txt', 'calendar.txt', 'calendar_dates.txt', 'agency.txt']
        dfs_in = {f: get_df(zip_path, f) for f in files_to_load}
        dfs_out = {f: get_df(intermediate_zip, f) for f in files_to_load}

        # 1. STOPS
        print("\n🚏 STOPS")
        if dfs_in['stops.txt'] is not None and dfs_out['stops.txt'] is not None:
            count_in = len(dfs_in['stops.txt'])
            count_out = len(dfs_out['stops.txt'])
            print(f"  • Count: {count_in} -> {count_out} ({count_out - count_in})")

        # 2. ROUTES
        print("\n🚌 ROUTES")
        if dfs_in['routes.txt'] is not None and dfs_out['routes.txt'] is not None:
            count_in = len(dfs_in['routes.txt'])
            count_out = len(dfs_out['routes.txt'])
            print(f"  • Count: {count_in} -> {count_out} ({count_out - count_in})")

        # 3. TRIPS
        print("\n🔄 TRIPS")
        if dfs_in['trips.txt'] is not None and dfs_out['trips.txt'] is not None:
            count_in = len(dfs_in['trips.txt'])
            count_out = len(dfs_out['trips.txt'])
            print(f"  • Count: {count_in} -> {count_out} ({count_out - count_in})")

        # 4. SHAPES (Flags: -m, -s)
        print("\nwv SHAPES (Remeasure & Simplify)")
        if dfs_in['shapes.txt'] is not None and dfs_out['shapes.txt'] is not None:
            pts_in = len(dfs_in['shapes.txt'])
            pts_out = len(dfs_out['shapes.txt'])
            ids_in = dfs_in['shapes.txt']['shape_id'].nunique()
            ids_out = dfs_out['shapes.txt']['shape_id'].nunique()
            print(f"  • Unique Shapes: {ids_in} -> {ids_out} ({ids_out - ids_in})")
            print(f"  • Total Points: {pts_in} -> {pts_out} ({pts_out - pts_in})")
            if pts_in > 0:
                print(f"  • Reduction: {((pts_in - pts_out) / pts_in) * 100:.1f}% fewer points")

        # 5. CALENDAR/SERVICES (Flags: -c)
        print("\n📅 SERVICES (Calendar Compression)")
        def get_service_ids(dfs):
            s_ids = set()
            if dfs['calendar.txt'] is not None:
                s_ids.update(dfs['calendar.txt']['service_id'])
            if dfs['calendar_dates.txt'] is not None:
                s_ids.update(dfs['calendar_dates.txt']['service_id'])
            return s_ids

        s_in = get_service_ids(dfs_in)
        s_out = get_service_ids(dfs_out)
        print(f"  • Active Service IDs: {len(s_in)} -> {len(s_out)} ({len(s_out) - len(s_in)})")

        # 6. ID CHANGES (Flags: -i)
        print("\n🆔 ID CHANGES (Renumbering)")
        # Just check one file to see if IDs look different
        if dfs_in['agency.txt'] is not None and dfs_out['agency.txt'] is not None:
            sample_in = dfs_in['agency.txt']['agency_id'].iloc[0] if not dfs_in['agency.txt'].empty else "N/A"
            sample_out = dfs_out['agency.txt']['agency_id'].iloc[0] if not dfs_out['agency.txt'].empty else "N/A"
            print(f"  • Agency ID Sample: '{sample_in}' -> '{sample_out}'")
            if sample_in != sample_out:
                print("    ✅ IDs have been renumbered.")
            else:
                print("    ℹ️ IDs appear unchanged (or were already simple).")

    except Exception as e:
        print(f"⚠️ Analysis failed: {e}")
        import traceback
        traceback.print_exc()

else:
    print(f"\n❌ gtfstidy modification failed with return code {result_mod.returncode}")

### 6.3 Optimization Phase
This step runs `gtfstidy` with aggressive optimization flags and writes the final `_tidy` directory.
- **Goal**: Reduce file size, remove redundancy, and improve map visualization.
- **Flags used in this notebook**: `--delete-orphans`, `--drop-errs`, `-O`, `-S`, `-R`, `-C`, `-T`, `--red-stops-fuzzy`, `--snap-stops`, `-W`.
- **Optional**: `-P` (redundant stop removal) is commented out by default (see code cell) because it can remove many non-redundant stops.

In [ ]:
print("🚀 3. OPTIMIZATION: Running gtfstidy (Uppercase Flags)...")

# Input is the output from the previous cell
intermediate_zip = str(MERGED_FEED_DIR) + "_modified.zip"
tidy_output_dir = str(MERGED_TIDY_FEED_DIR)

# Run gtfstidy with UPPERCASE flags (aggressive optimization/deduplication)
# --delete-orphans: remove unreferenced entities
# --drop-errs: drop entries that fail validation
# -O: (gtfstidy) optimize feed (general)
# -S: remove redundant shapes
# -R: remove redundant routes
# -C: remove service duplicates / compress services
# -T: minimize frequencies/stop_times/trips
# -P: remove redundant stops (commented out below; see cmd list)
# --red-stops-fuzzy: fuzzy stop deduplication
# --snap-stops: snap stops to shapes (improves map visualization)
# -W: show warnings / details about dropped entries

# Remove existing output to avoid stale results
if os.path.isdir(tidy_output_dir):
    shutil.rmtree(tidy_output_dir)

cmd = [
    'gtfstidy',
    '--delete-orphans',
    '--drop-errs',
    '-O',
    '-S',
    '-R',
    '-C',
    '-T',
    #'-P', removing many non-redundant stops
    '--red-stops-fuzzy',
    '--snap-stops',
    '-W',
    '-o',
    tidy_output_dir,
    intermediate_zip,
]

print(f"Command: {subprocess.list2cmdline(cmd)}\n")

try:
    result = subprocess.run(cmd, capture_output=True, text=True)
except FileNotFoundError:
    raise RuntimeError("gtfstidy not found on PATH. Install it or add it to PATH, then re-run this cell.")

# Print output
if result.stdout:
    print("Output:")
    print(result.stdout)
if result.stderr:
    print("Errors/Warnings:")
    print(result.stderr)

print(f"\nReturn code: {result.returncode}")

if result.returncode == 0:
    print(f"\n✅ SUCCESS! GTFS feed optimized and tidied!")
    print(f"📁 Output directory: {tidy_output_dir}")

    print("\n📊 DETAILED OPTIMIZATION REPORT")
    print("==============================")

    try:
        import zipfile
        
        def get_df(path, filename):
            """Load a GTFS table as strings, preserving blanks."""
            p = Path(path)
            if p.is_file() and p.suffix == '.zip':
                with zipfile.ZipFile(p) as z:
                    if filename in z.namelist():
                        return pd.read_csv(
                            z.open(filename),
                            dtype=str,
                            keep_default_na=False,
                            na_filter=False,
                        )
            elif p.is_dir():
                f = p / filename
                if f.exists():
                    return pd.read_csv(
                        f,
                        dtype=str,
                        keep_default_na=False,
                        na_filter=False,
                    )
            return None

        # Load Dataframes
        files_to_load = ['stops.txt', 'routes.txt', 'trips.txt', 'shapes.txt', 'calendar.txt', 'calendar_dates.txt', 'agency.txt']
        dfs_in = {f: get_df(intermediate_zip, f) for f in files_to_load}
        dfs_out = {f: get_df(tidy_output_dir, f) for f in files_to_load}

        # 1. STOPS (Flags: --red-stops-fuzzy, --snap-stops)
        print("\n🚏 STOPS (Deduplication & Snapping)")
        if dfs_in['stops.txt'] is not None and dfs_out['stops.txt'] is not None:
            count_in = len(dfs_in['stops.txt'])
            count_out = len(dfs_out['stops.txt'])
            print(f"  • Count: {count_in} -> {count_out} ({count_out - count_in})")
            
            # Snapping check (robust to blanks)
            stops_in = dfs_in['stops.txt'].copy()
            stops_out = dfs_out['stops.txt'].copy()
            for d in (stops_in, stops_out):
                for c in ['stop_lat', 'stop_lon']:
                    if c in d.columns:
                        d[c] = pd.to_numeric(d[c], errors='coerce')

            common = pd.merge(stops_in, stops_out, on='stop_id', suffixes=('_in', '_out'))
            # Only count rows with valid coordinates in both
            valid = common['stop_lat_in'].notna() & common['stop_lon_in'].notna() & common['stop_lat_out'].notna() & common['stop_lon_out'].notna()
            moved = (
                (common.loc[valid, 'stop_lat_in'] - common.loc[valid, 'stop_lat_out']).abs().gt(1e-6)
                | (common.loc[valid, 'stop_lon_in'] - common.loc[valid, 'stop_lon_out']).abs().gt(1e-6)
            )
            print(f"  • Locations Moved: {int(moved.sum())} stops snapped to shapes.")

        # 2. ROUTES (Flags: -R)
        print("\n🚌 ROUTES (Redundant Route Removal)")
        if dfs_in['routes.txt'] is not None and dfs_out['routes.txt'] is not None:
            count_in = len(dfs_in['routes.txt'])
            count_out = len(dfs_out['routes.txt'])
            print(f"  • Count: {count_in} -> {count_out} ({count_out - count_in})")

        # 3. TRIPS (Flags: -T, --drop-errs)
        print("\n🔄 TRIPS (Optimization & Error Dropping)")
        if dfs_in['trips.txt'] is not None and dfs_out['trips.txt'] is not None:
            count_in = len(dfs_in['trips.txt'])
            count_out = len(dfs_out['trips.txt'])
            print(f"  • Count: {count_in} -> {count_out} ({count_out - count_in})")

        # 4. SHAPES (Flags: -S)
        print("\nwv SHAPES (Simplification)")
        if dfs_in['shapes.txt'] is not None and dfs_out['shapes.txt'] is not None:
            pts_in = len(dfs_in['shapes.txt'])
            pts_out = len(dfs_out['shapes.txt'])
            ids_in = dfs_in['shapes.txt']['shape_id'].nunique()
            ids_out = dfs_out['shapes.txt']['shape_id'].nunique()
            print(f"  • Unique Shapes: {ids_in} -> {ids_out} ({ids_out - ids_in})")
            print(f"  • Total Points: {pts_in} -> {pts_out} ({pts_out - pts_in})")
            if pts_in > 0:
                print(f"  • Reduction: {((pts_in - pts_out) / pts_in) * 100:.1f}% fewer points")

        # 5. CALENDAR/SERVICES (Flags: -C)
        print("\n📅 SERVICES (Calendar Compression)")
        # Count unique service_ids across calendar and calendar_dates
        def get_service_ids(dfs):
            s_ids = set()
            if dfs['calendar.txt'] is not None:
                s_ids.update(dfs['calendar.txt']['service_id'])
            if dfs['calendar_dates.txt'] is not None:
                s_ids.update(dfs['calendar_dates.txt']['service_id'])
            return s_ids

        s_in = get_service_ids(dfs_in)
        s_out = get_service_ids(dfs_out)
        print(f"  • Active Service IDs: {len(s_in)} -> {len(s_out)} ({len(s_out) - len(s_in)})")

        # 6. ORPHANS / DELETED IDs (Flags: --delete-orphans)
        print("\n🗑️ DELETED IDs (Orphans & Errors)")
        for filename in ['agency.txt', 'routes.txt', 'trips.txt', 'stops.txt', 'shapes.txt']:
            if dfs_in[filename] is None or dfs_out[filename] is None:
                continue

            col = filename.replace('.txt', '_id')
            if col == 'shapes_id':
                col = 'shape_id'
            if col == 'routes_id':
                col = 'route_id'
            if col == 'trips_id':
                col = 'trip_id'
            if col == 'stops_id':
                col = 'stop_id'

            if col not in dfs_in[filename].columns or col not in dfs_out[filename].columns:
                print(f"  • {filename}: can't compute deletions (missing '{col}')")
                continue

            # Avoid counting blank IDs as deleted
            ids_in = set(x for x in dfs_in[filename][col].astype(str) if x)
            ids_out = set(x for x in dfs_out[filename][col].astype(str) if x)
            deleted = ids_in - ids_out
            if deleted:
                examples = ', '.join(sorted(deleted)[:3])
                print(f"  • {filename}: {len(deleted)} deleted. Examples: {examples}...")

    except Exception as e:
        print(f"⚠️ Analysis failed: {e}")
        import traceback
        traceback.print_exc()

else:
    print(f"\n❌ gtfstidy failed with return code {result.returncode}")
    print(f"\n💡 Check the validation report in the previous cell for details.")

### 6.4 Cleanup

This optional step **removes intermediate artifacts** created during the workflow.

It will delete:
- `Merged_Feed/`
- `Merged_Feed.zip`
- `Merged_Feed_modified.zip`
- `Unique_Feeds/`

**Warning:** This is a permanent delete. Double-check that `MERGED_FEED_DIR` and `UNIQUE_FEEDS_DIR` point to the intended workspace before running.


In [ ]:
print("\nCleaning up intermediate artifacts (Merged_Feed + Unique_Feeds)...")

cleanup_targets = [
    ("directory", "Merged_Feed", Path(MERGED_FEED_DIR)),
    ("file", "Merged_Feed.zip", Path(str(MERGED_FEED_DIR) + ".zip")),
    ("file", "Merged_Feed_modified.zip", Path(str(MERGED_FEED_DIR) + "_modified.zip")),
    ("directory", "Unique_Feeds", Path(UNIQUE_FEEDS_DIR)),
]

for kind, label, path in cleanup_targets:
    try:
        if not path.exists():
            print(f"Skipped {label} (not found)")
            continue

        if path.is_file():
            path.unlink()
            print(f"Deleted {label}")
        else:
            shutil.rmtree(path)
            print(f"Deleted {label} folder")
    except Exception as exc:
        print(f"Warning: could not delete {label}: {exc}")


## 7. Calculate Weekly Trip Frequency

Calculate weekly trips per shape for the merged feed. Load trips, calendar, and routes; filter trips with shape_id; compute weekly days per service; group by shape_id to sum contributions; include agency details; and export to CSV.

In [ ]:
def load_gtfs(gtfs_path):
    """
    Load GTFS files: routes.txt, trips.txt, calendar.txt, and agency.txt
    """
    encodings_to_try = ['utf-8-sig', 'utf-16', 'utf-8']
    
    def read_csv_with_fallback(file_path):
        for encoding in encodings_to_try:
            try:
                return pd.read_csv(file_path, encoding=encoding)
            except UnicodeDecodeError:
                continue
        raise UnicodeDecodeError(f"Could not decode {file_path} with any of {encodings_to_try}")
    
    routes = read_csv_with_fallback(os.path.join(gtfs_path, 'routes.txt'))
    trips = read_csv_with_fallback(os.path.join(gtfs_path, 'trips.txt'))
    calendar = read_csv_with_fallback(os.path.join(gtfs_path, 'calendar.txt'))
    agency = read_csv_with_fallback(os.path.join(gtfs_path, 'agency.txt'))
    return routes, trips, calendar, agency
def calculate_weekly_trips_per_shape(routes, trips, calendar, agency):
    """
    Calculate weekly trips for all shapes, based on active days per service.
    Multiple trip_ids can share the same service_id and shape_id.
    Only includes trips that have a shape_id.
    Returns one row per unique shape_id, with weekly_trips and lists of associated IDs.
    """
    # Filter trips to only those with shape_id
    trips = trips.dropna(subset=['shape_id'])
    
    # Merge trips with routes to get agency_id
    trips = trips.merge(routes[['route_id', 'agency_id']], on='route_id', how='left')
    
    # Calculate weekly days for each service: sum of active days
    calendar['weekly_days'] = calendar[['monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday']].sum(axis=1)
    
    # Merge trips with calendar to get weekly_days
    trips = trips.merge(calendar[['service_id', 'weekly_days']], on='service_id', how='left')
    
    # Each trip contributes weekly_days to its shape
    trips['contribution'] = trips['weekly_days']
    
    # Sum contributions per shape
    weekly_trips = trips.groupby('shape_id')['contribution'].sum().reset_index()
    
    # Get unique associated IDs per shape
    unique_agencies = trips.groupby('shape_id')['agency_id'].unique().reset_index().rename(columns={'agency_id': 'agency_ids'})
    unique_services = trips.groupby('shape_id')['service_id'].unique().reset_index().rename(columns={'service_id': 'service_ids'})
    unique_routes = trips.groupby('shape_id')['route_id'].unique().reset_index().rename(columns={'route_id': 'route_ids'})
    unique_trips = trips.groupby('shape_id')['trip_id'].unique().reset_index().rename(columns={'trip_id': 'trip_ids'})
    
    # Merge all into weekly_trips
    weekly_trips = weekly_trips.merge(unique_agencies, on='shape_id', how='left')
    weekly_trips = weekly_trips.merge(unique_services, on='shape_id', how='left')
    weekly_trips = weekly_trips.merge(unique_routes, on='shape_id', how='left')
    weekly_trips = weekly_trips.merge(unique_trips, on='shape_id', how='left')
    
    # Since each shape has unique agency_id, extract the single value
    weekly_trips['agency_id'] = weekly_trips['agency_ids'].apply(lambda x: x[0] if len(x) > 0 else None)
    weekly_trips.drop(columns=['agency_ids'], inplace=True)
    
    # Merge with agency to get agency_name
    weekly_trips = weekly_trips.merge(agency[['agency_id', 'agency_name']], on='agency_id', how='left')
    
    # Rename contribution to weekly_trips
    weekly_trips.rename(columns={'contribution': 'weekly_trips'}, inplace=True)
    
    return weekly_trips
# Example usage
# Set the path to the combined GTFS folder
gtfs_path = str(MERGED_TIDY_FEED_DIR)

agency_name = 'All_Agencies'

print(f"Processing {agency_name}...")

try:
    # Load the data
    routes, trips, calendar, agency = load_gtfs(gtfs_path)
    
    # Calculate weekly trips per shape
    result = calculate_weekly_trips_per_shape(routes, trips, calendar, agency)
    
    print(f"Completed {agency_name}")
    
    # Display the result
    print(result)
    
    # Save to CSV
    csv_path = str(WEEKLY_TRIPS_CSV)
    result.to_csv(csv_path, index=False)
    print(f"Saved results to {csv_path}")
    
except Exception as e:
    print(f"Error processing {agency_name}: {e}")

# 8. Build Custom GeoJSON from GTFS

This approach gives you complete control over which properties are included in the GeoJSON output.

### 8.1 Load GTFS Data and Supporting Files

This cell loads all required GTFS files from the merged and tidied feed directory, along with supporting data files:

- **Core GTFS files**: Reads `agency.txt`, `routes.txt`, `trips.txt`, `stops.txt`, `shapes.txt`, and `stop_times.txt` from the merged feed
- **Weekly trips data**: Loads the `WEEKLY_TRIPS_CSV` file generated in Step 7 (configured in the Configuration section)

The cell prints summary statistics showing the number of routes, stops, and shape points loaded, along with the available columns in the routes and stops dataframes. This helps verify that all required data is present before building the GeoJSON features.

In [ ]:
# Define the GTFS directory
gtfs_dir = str(MERGED_TIDY_FEED_DIR)

# Read GTFS files
print("Reading GTFS files...")
agency = pd.read_csv(f"{gtfs_dir}/agency.txt")
routes = pd.read_csv(f"{gtfs_dir}/routes.txt")
trips = pd.read_csv(f"{gtfs_dir}/trips.txt")
stops = pd.read_csv(f"{gtfs_dir}/stops.txt")
shapes = pd.read_csv(f"{gtfs_dir}/shapes.txt")
stop_times = pd.read_csv(f"{gtfs_dir}/stop_times.txt")

# Read weekly trips data from Step 7 output
weekly_trips_df = pd.read_csv(WEEKLY_TRIPS_CSV)
print(f"Loaded weekly trips data for {len(weekly_trips_df)} shapes")

print(f"Routes columns: {routes.columns.tolist()}")
print(f"Stops columns: {stops.columns.tolist()}")

### 8.2 Build Route Features as LineString Geometry

This cell creates GeoJSON LineString features for each route-shape combination in the GTFS feed:

1. **Group and sort shape coordinates**: 
   - Sorts `shapes.txt` by `shape_id` and `shape_pt_sequence` to ensure proper coordinate ordering
   - Groups coordinates by `shape_id` to create arrays of `[longitude, latitude]` pairs (GeoJSON requires lon/lat order)

2. **Extract route-shape relationships**:
   - Gets unique combinations of `route_id` and `shape_id` from `trips.txt` (handles routes with multiple shapes, e.g., different AM/PM service patterns)
   - Merges with `routes.txt` and `agency.txt` to enrich with route and agency metadata

3. **Build route feature properties**:
   - **Core fields**: `agency_name`, `agency_id`, `route_id`, `route_long_name`
   - **Weekly trips**: Joins weekly trip frequency data from the CSV file using `shape_id`
   - **Stop information**: Extracts an ordered list of `stop_id`s for each route by sampling a representative trip's stop sequence

4. **Create GeoJSON features**: Each feature contains a LineString geometry with the shape coordinates and a properties dictionary with route metadata.

The output is a list of route features that will be combined with stop features to create the final GeoJSON FeatureCollection.

In [ ]:
# Build route shapes as LineString features
print("Building route features...")

# Group shapes by shape_id and sort by sequence
shapes_sorted = shapes.sort_values(['shape_id', 'shape_pt_sequence'])
shape_coords = shapes_sorted.groupby('shape_id').apply(
    lambda x: x[['shape_pt_lon', 'shape_pt_lat']].values.tolist()
).to_dict()

# Get unique route-shape combinations from trips
route_shapes = trips[['route_id', 'shape_id']].drop_duplicates()

# Merge with route information
route_shapes = route_shapes.merge(routes, on='route_id', how='left')
route_shapes = route_shapes.merge(agency, on='agency_id', how='left')

# Create route features
route_features = []
for _, row in route_shapes.iterrows():
    shape_id = row['shape_id']
    route_id = row['route_id']
    agency_id = row.get('agency_id', '')
    if pd.notna(shape_id) and shape_id in shape_coords:
        coords = shape_coords[shape_id]

        # Build custom properties - add or remove fields as needed
        properties = {
            'feature_type': 'route',
            'agency_name': row.get('agency_name', ''),
            'agency_id': agency_id,
            'route_id': route_id,
            'route_long_name': row.get('route_long_name', ''),
        }

        # Add weekly trips from CSV file
        weekly_trips = weekly_trips_df[weekly_trips_df['shape_id'] == shape_id]['weekly_trips'].iloc[0] if len(weekly_trips_df[weekly_trips_df['shape_id'] == shape_id]) > 0 else 0
        properties['weekly_trips'] = int(weekly_trips)

        # Add stops for this route (ordered by sequence, keeping duplicates)
        route_stops = []
        # Get all trips for this route that use this shape
        route_trips = trips[(trips['route_id'] == route_id) & (trips['shape_id'] == shape_id)]
        if len(route_trips) > 0:
            # Get one representative trip (first one) to get stop sequence
            sample_trip_id = route_trips.iloc[0]['trip_id']
            trip_stop_times = stop_times[stop_times['trip_id'] == sample_trip_id].sort_values('stop_sequence')
            route_stops = trip_stop_times['stop_id'].tolist()

        properties['stops'] = route_stops
        properties['stop_count'] = len(route_stops)

        feature = {
            'type': 'Feature',
            'properties': properties,
            'geometry': {
                'type': 'LineString',
                'coordinates': coords
            }
        }
        route_features.append(feature)

print(f"Created {len(route_features)} route features")

### 8.3 Build Stop Features as Point Geometry

This cell creates GeoJSON Point features for each stop in the GTFS feed, enriched with information about which routes serve each stop:

1. **Link stops to routes**:
   - Merges `stop_times.txt` with `trips.txt` to connect stops to routes via `trip_id`
   - Further merges with `routes.txt` and `agency.txt` to get route and agency details
   - Groups by `stop_id` to collect all unique routes serving each stop

2. **Build stop-route relationships**:
   - For each stop, creates a list of route information dictionaries containing:
     - `route_id`, `agency_id`, `route_long_name`
   - Only adds unique routes to avoid duplicates

3. **Create stop feature properties**:
   - **Core fields**: `stop_id`, `stop_name`, `agency_name` (from first route serving the stop)
   - **Route information**: `routes` array containing all routes serving this stop, and `route_count` (useful for identifying major transfer hubs)

4. **Create GeoJSON features**: Each feature contains a Point geometry with `[longitude, latitude]` coordinates and a properties dictionary with stop metadata.

The output is a list of stop features that will be combined with route features to create the final GeoJSON FeatureCollection.

In [ ]:
# Build stop features with routes information
print("Building stop features...")

# Get all routes that serve each stop
stop_routes = stop_times.merge(trips[['trip_id', 'route_id']], on='trip_id')
stop_routes = stop_routes.merge(routes, on='route_id')
stop_routes = stop_routes.merge(agency, on='agency_id', how='left')

# Group routes by stop
stop_route_info = defaultdict(list)
for _, row in stop_routes.iterrows():
    stop_id = row['stop_id']
    route_info = {
        'route_id': row['route_id'],
        'agency_id': row.get('agency_id', ''),
    }

    # Only add unique routes
    if route_info not in stop_route_info[stop_id]:
        stop_route_info[stop_id].append(route_info)

# Create stop features
stop_features = []
for _, stop in stops.iterrows():
    stop_id = stop['stop_id']

    # Get agency name from the first route serving this stop
    agency_name = ''
    if stop_id in stop_route_info and len(stop_route_info[stop_id]) > 0:
        first_route = stop_route_info[stop_id][0]
        agency_id = first_route.get('agency_id')
        if agency_id:
            agency_info = agency[agency['agency_id'] == agency_id]
            if len(agency_info) > 0:
                agency_name = agency_info.iloc[0]['agency_name']

    # Build custom properties
    properties = {
        'feature_type': 'stop',
        'stop_id': stop_id,
        'stop_name': stop.get('stop_name', ''),
        'agency_name': agency_name,
        'routes': stop_route_info.get(stop_id, [])
    }

    # Add number of serving routes
    properties['route_count'] = int(len(stop_route_info.get(stop_id, [])))

    feature = {
        'type': 'Feature',
        'properties': properties,
        'geometry': {
            'type': 'Point',
            'coordinates': [float(stop['stop_lon']), float(stop['stop_lat'])]
        }
    }
    stop_features.append(feature)

print(f"Created {len(stop_features)} stop features")

### 8.4 Combine Features and Export GeoJSON

This cell combines all route and stop features into a single GeoJSON FeatureCollection and saves it to a file:

1. **Create FeatureCollection**:
   - Combines the `route_features` list (LineString features) with the `stop_features` list (Point features)
   - Wraps them in a GeoJSON FeatureCollection structure with `type: "FeatureCollection"` and a `features` array

2. **Export to file**:
   - Saves the combined GeoJSON to `GEOJSON_OUTPUT` (configured in the Configuration section)
   - Uses UTF-8 encoding to handle special characters in agency names, route names, and stop names
   - Formats with indentation (2 spaces) for readability

3. **Summary output**:
   - Prints the total number of features created (routes + stops)
   - Provides a breakdown showing how many route features and stop features are included

The resulting GeoJSON file can be imported into GIS tools (QGIS, ArcGIS Pro), web mapping libraries (Leaflet, Mapbox GL JS), or used directly in the Folium visualization workflow in Step 9.

In [ ]:
# Combine all features and create GeoJSON
print("Creating final GeoJSON...")

# Create combined GeoJSON
geojson = {
    'type': 'FeatureCollection',
    'features': route_features + stop_features
}

# Defensive cleanup: ensure unwanted keys are not written
UNWANTED_PROPERTY_KEYS = {
    'route_type',
    'route_color',
    'route_text_color',
    'zone_id',
}

for feat in geojson.get('features', []):
    props = feat.get('properties') or {}

    for k in UNWANTED_PROPERTY_KEYS:
        props.pop(k, None)

    # Also clean nested stop->routes route_info dictionaries
    routes_list = props.get('routes')
    if isinstance(routes_list, list):
        for r in routes_list:
            if isinstance(r, dict):
                for k in UNWANTED_PROPERTY_KEYS:
                    r.pop(k, None)

# Save combined file
output_file = Path(GEOJSON_OUTPUT).resolve()
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(geojson, f, indent=2)

print(f"Combined GeoJSON saved to {output_file}")
print(f"Total features: {len(geojson['features'])} ({len(route_features)} routes + {len(stop_features)} stops)")

### 8.5 Preview Sample Features

This cell provides a quick preview of the generated GeoJSON features to verify the structure and content:

1. **Sample route feature**:
   - Displays the first route feature from the GeoJSON (truncated to 500 characters for readability)
   - Shows the complete structure including geometry type (LineString), coordinates, and all properties
   - Useful for verifying that route metadata (agency name, route names, weekly trips, source information) is correctly included

2. **Sample stop feature**:
   - Displays the first stop feature (located after all route features in the features array)
   - Shows the Point geometry with coordinates and stop properties
   - Verifies that route relationships, agency information, and optional fields are properly structured

This preview helps ensure the GeoJSON output is correctly formatted before using it in GIS tools or web mapping applications. The full structure can be inspected by opening the saved `.geojson` file in a text editor or JSON viewer.


In [ ]:
# Preview a sample route feature
print("Sample Route Feature:")
print(json.dumps(geojson['features'][0], indent=2)[:500])

print("\n" + "="*50 + "\n")

# Preview a sample stop feature
stop_feature = None
stop_feature_index = len(route_features)  # First stop after all routes
if stop_feature_index < len(geojson['features']):
    stop_feature = geojson['features'][stop_feature_index]

if stop_feature:
    print("Sample Stop Feature:")
    print(json.dumps(stop_feature, indent=2)[:500])

# 9. Interactive Web Map

Build an interactive Folium web map from the GeoJSON created in Step 8.

- Loads `GEOJSON_OUTPUT` (routes + stops) and separates LineString vs Point features.
- Styles routes by agency (distinct colors) and uses line thickness to encode `weekly_trips`.
- Adds stop markers (with optional clustering) and a LayerControl so agencies can be toggled on/off.
- Saves the final HTML map to `MAP_OUTPUT`.

In [ ]:
# Custom stop markers: hollow circle with bus icon
STOP_USE_MARKER_CLUSTER = False        # cluster stops per agency layer
STOP_CLUSTER_DISABLE_AT_ZOOM = 12      # stop clustering at this zoom

# Load GeoJSON produced in Step 8
geojson_file = str(Path(GEOJSON_OUTPUT).resolve())
print(f"Loading GeoJSON from {geojson_file}")
with open(geojson_file, 'r', encoding='utf-8') as f:
    geojson_data = json.load(f)

OUTPUT_MAP = str(Path(MAP_OUTPUT).resolve())
print(f"Saving map to {OUTPUT_MAP}")

import colorsys
from folium import DivIcon

def generate_distinct_colors(n):
    colors = []
    for i in range(n):
        hue = (i * 137.5) % 360  # golden angle for distinct hues
        rgb = colorsys.hsv_to_rgb(hue/360, 0.5, 0.5)
        color = '#%02x%02x%02x' % (int(rgb[0]*255), int(rgb[1]*255), int(rgb[2]*255))
        colors.append(color)
    return colors

# Helper function to format properties as HTML
def format_properties_html(props):
    html = ""
    for key, value in props.items():
        if value in (None, '', [], {}):
            continue
        formatted_key = key.replace('_', ' ').title()
        if isinstance(value, list):
            if len(value) > 0:
                if isinstance(value[0], dict):
                    formatted_value = "<br>&nbsp;&nbsp;• " + "<br>&nbsp;&nbsp;• ".join([
                        ", ".join([f"{k}: {v}" for k, v in item.items() if v])
                        for item in value[:5]
                    ])
                    if len(value) > 5:
                        formatted_value += f"<br>&nbsp;&nbsp;... and {len(value) - 5} more"
                else:
                    formatted_value = ", ".join([str(v) for v in value[:10]])
                    if len(value) > 10:
                        formatted_value += f", ... and {len(value) - 10} more"
            else:
                continue
        elif isinstance(value, dict):
            formatted_value = "<br>&nbsp;&nbsp;• " + "<br>&nbsp;&nbsp;• ".join([
                f"{k}: {v}" for k, v in value.items() if v
            ])
        else:
            formatted_value = str(value)
        html += f"<b>{formatted_key}:</b> {formatted_value}<br>"
    return html

# Line weight by weekly trips
def get_line_weight(weekly_trips):
    if weekly_trips < 6:
        return 2, 1
    elif weekly_trips < 10:
        return 4, 2
    elif weekly_trips < 20:
        return 6, 3
    elif weekly_trips < 35:
        return 8, 4
    else:
        return 10, 5

# Folium map
map_center = [39.8283, -98.5795]
m = folium.Map(location=map_center, zoom_start=5, tiles='OpenStreetMap')

# Group features by agency and prepare metadata
agency_features = defaultdict(list)
all_features_with_metadata = []

for feature in geojson_data['features']:
    agency = feature['properties'].get('agency_name', 'Unknown')
    agency_features[agency].append(feature)

agencies = list(agency_features.keys())
colors_list = generate_distinct_colors(len(agencies))
agency_color_map = {agency: color for agency, color in zip(agencies, colors_list)}

feature_id = 0
for agency, features in agency_features.items():
    color = agency_color_map[agency]
    for feature in features:
        all_features_with_metadata.append({
            'id': feature_id,
            'feature': feature,
            'agency': agency,
            'color': color
        })
        feature_id += 1

for agency, features in agency_features.items():
    group = FeatureGroup(name=agency, show=False)
    color = agency_color_map[agency]
    
    # Optional stop clustering per agency
    stop_container = group
    if STOP_USE_MARKER_CLUSTER:
        stop_container = MarkerCluster(control=False, disableClusteringAtZoom=STOP_CLUSTER_DISABLE_AT_ZOOM)
        stop_container.add_to(group)
    
    for feature in features:
        if feature['geometry']['type'] == 'LineString':
            coords = feature['geometry']['coordinates']
            latlon = [[lat, lon] for lon, lat in coords]
            props = feature['properties']
            weekly_trips = props.get('weekly_trips', 0)
            weight, category = get_line_weight(weekly_trips)
            popup_html = format_properties_html(props)
            PolyLine(latlon, color=color, weight=weight, opacity=0.8,
                    popup=Popup(popup_html, max_width=400, max_height=300)).add_to(group)
        elif feature['geometry']['type'] == 'Point':
            lon, lat = feature['geometry']['coordinates']
            props = feature['properties']
            popup_html = format_properties_html(props)
            # Custom hollow circle with bus icon
            html = f'''
            <div style="border: 2px solid {color}; border-radius: 50%; width: 20px; height: 20px; display: flex; align-items: center; justify-content: center; background: white;">
                <i class="fa fa-bus" style="color: {color}; font-size: 12px;"></i>
            </div>
            '''
            icon = DivIcon(html=html, icon_size=(20,20), icon_anchor=(10,10))
            folium.Marker([lat, lon], icon=icon,
                          popup=Popup(popup_html, max_width=400, max_height=300)).add_to(stop_container)
    group.add_to(m)

# Add JavaScript for overlapping feature navigation (unchanged)
overlap_handler_js = """
<script>
var allFeaturesData = """ + json.dumps([{
    'id': f['id'],
    'geometry': f['feature']['geometry'],
    'properties': f['feature']['properties'],
    'agency': f['agency'],
    'color': f['color']
} for f in all_features_with_metadata]) + """;
var currentOverlapIndex = 0;
var overlappingFeatures = [];
var activePopup = null;
function haversineDistance(lat1, lon1, lat2, lon2) {
    const R = 6371000;
    const dLat = (lat2 - lat1) * Math.PI / 180;
    const dLon = (lon2 - lon1) * Math.PI / 180;
    const a = Math.sin(dLat/2) * Math.sin(dLat/2) +
              Math.cos(lat1 * Math.PI / 180) * Math.cos(lat2 * Math.PI / 180) *
              Math.sin(dLon/2) * Math.sin(dLon/2);
    const c = 2 * Math.atan2(Math.sqrt(a), Math.sqrt(1-a));
    return R * c;
}
function pointToSegmentDistance(px, py, x1, y1, x2, y2) {
    const A = px - x1;
    const B = py - y1;
    const C = x2 - x1;
    const D = y2 - y1;
    const dot = A * C + B * D;
    const lenSq = C * C + D * D;
    let param = -1;
    if (lenSq !== 0) param = dot / lenSq;
    let xx, yy;
    if (param < 0) { xx = x1; yy = y1; }
    else if (param > 1) { xx = x2; yy = y2; }
    else { xx = x1 + param * C; yy = y1 + param * D; }
    return haversineDistance(px, py, xx, yy);
}
function findOverlappingFeatures(clickLat, clickLng, threshold = 100) {
    const nearby = [];
    allFeaturesData.forEach(function(featureData) {
        const geom = featureData.geometry;
        let isNearby = false;
        if (geom.type === 'LineString') {
            for (let i = 0; i < geom.coordinates.length - 1; i++) {
                const [lon1, lat1] = geom.coordinates[i];
                const [lon2, lat2] = geom.coordinates[i + 1];
                const dist = pointToSegmentDistance(clickLat, clickLng, lat1, lon1, lat2, lon2);
                if (dist <= threshold) { isNearby = true; break; }
            }
        } else if (geom.type === 'Point') {
            const [lon, lat] = geom.coordinates;
            const dist = haversineDistance(clickLat, clickLng, lat, lon);
            if (dist <= threshold) { isNearby = true; }
        }
        if (isNearby) { nearby.push(featureData); }
    });
    return nearby;
}
function formatPropertiesHtml(props) {
    let html = '';
    for (const [key, value] of Object.entries(props)) {
        if (value === null || value === '' || (Array.isArray(value) && value.length === 0) || (typeof value === 'object' && !Array.isArray(value) && Object.keys(value).length === 0)) { continue; }
        const formattedKey = key.replace(/_/g, ' ').replace(/\\b\\w/g, l => l.toUpperCase());
        let formattedValue;
        if (Array.isArray(value)) {
            if (value.length > 0) {
                if (typeof value[0] === 'object') {
                    formattedValue = '<br>&nbsp;&nbsp;• ' + value.slice(0, 5).map(item => Object.entries(item).filter(([k,v]) => v).map(([k,v]) => k + ': ' + v).join(', ')).join('<br>&nbsp;&nbsp;• ');
                    if (value.length > 5) formattedValue += '<br>&nbsp;&nbsp;... and ' + (value.length - 5) + ' more';
                } else {
                    formattedValue = value.slice(0, 10).join(', ');
                    if (value.length > 10) formattedValue += ', ... and ' + (value.length - 10) + ' more';
                }
            } else { continue; }
        } else if (typeof value === 'object') {
            formattedValue = '<br>&nbsp;&nbsp;• ' + Object.entries(value).filter(([k,v]) => v).map(([k,v]) => k + ': ' + v).join('<br>&nbsp;&nbsp;• ');
        } else {
            formattedValue = String(value);
        }
        html += '<b>' + formattedKey + ':</b> ' + formattedValue + '<br>';
    }
    return html;
}
function showFeaturePopup(featureData, index, total, latlng) {
    const props = featureData.properties;
    let popupContent = '<div style="max-width: 400px; max-height: 300px; overflow-y: auto;">';
    if (total > 1) {
        popupContent += '<div style="background-color: #f0f0f0; padding: 8px; margin: -10px -10px 10px -10px; border-bottom: 2px solid #ddd;">';
        popupContent += '<div style="display: flex; justify-content: space-between; align-items: center;">';
        popupContent += '<button onclick="navigateOverlap(-1)" style="cursor: pointer; padding: 5px 10px; background: #4CAF50; color: white; border: none; border-radius: 3px;">◀ Prev</button>';
        popupContent += '<span style="font-weight: bold;">Feature ' + (index + 1) + ' of ' + total + '</span>';
        popupContent += '<button onclick="navigateOverlap(1)" style="cursor: pointer; padding: 5px 10px; background: #4CAF50; color: white; border: none; border-radius: 3px;">Next ▶</button>';
        popupContent += '</div>';
        popupContent += '<div style="margin-top: 5px; font-size: 12px; color: #666;">Agency: ' + featureData.agency + '</div>';
        popupContent += '</div>';
    }
    popupContent += '</div>';
    return popupContent;
}
function navigateOverlap(direction) {
    currentOverlapIndex = (currentOverlapIndex + direction + overlappingFeatures.length) % overlappingFeatures.length;
    const featureData = overlappingFeatures[currentOverlapIndex];
    const popupContent = showFeaturePopup(featureData, currentOverlapIndex, overlappingFeatures.length, activePopup.getLatLng());
    activePopup.setContent(popupContent);
}
function selectAllLayers() {
    const layerControl = document.querySelector('.leaflet-control-layers');
    if (layerControl) {
        const inputs = layerControl.querySelectorAll('input[type=\"checkbox\"]');
        const allChecked = Array.from(inputs).every(input => input.checked);
        const newState = !allChecked;
        inputs.forEach(input => { input.checked = newState; input.dispatchEvent(new Event('change')); });
        const button = document.querySelector('button[onclick=\"selectAllLayers()\"]');
        if (button) { button.textContent = newState ? 'Deselect All' : 'Select All'; }
    }
}
setTimeout(function() {
    const mapElement = document.querySelector('.folium-map');
    if (!mapElement || !mapElement._leaflet_id) { console.log('Map not ready, waiting...'); return; }
    const map = window[mapElement._leaflet_id];
    map.on('click', function(e) {
        const clickedFeatures = findOverlappingFeatures(e.latlng.lat, e.latlng.lng, 100);
        if (clickedFeatures.length > 0) {
            overlappingFeatures = clickedFeatures;
            currentOverlapIndex = 0;
            const popupContent = showFeaturePopup(clickedFeatures[0], 0, clickedFeatures.length, e.latlng);
            activePopup = L.popup().setLatLng(e.latlng).setContent(popupContent).openOn(map);
        }
    });
}, 1000);
</script>
"""

m.get_root().html.add_child(folium.Element(overlap_handler_js))

legend_html = '''
<div style="position: fixed; bottom: 50px; left: 50px; width: 220px; height: 180px; background-color: white; border:2px solid grey; z-index:9999; font-size:14px; padding: 10px">
<p style="margin-top:0; font-weight:bold;">Weekly Trip Frequency</p>
<p style="margin:5px 0;"><span style="display:inline-block; width:30px; height:2px; background-color:black;"></span> &lt; 6 trips</p>
<p style="margin:5px 0;"><span style="display:inline-block; width:30px; height:4px; background-color:black;"></span> 6-10 trips</p>
<p style="margin:5px 0;"><span style="display:inline-block; width:30px; height:6px; background-color:black;"></span> 11-20 trips</p>
<p style="margin:5px 0;"><span style="display:inline-block; width:30px; height:8px; background-color:black;"></span> 21-35 trips</p>
<p style="margin:5px 0;"><span style="display:inline-block; width:30px; height:10px; background-color:black;"></span> 36+ trips</p>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

LayerControl(collapsed=False).add_to(m)
m.save(OUTPUT_MAP)
print(f"Map saved as {OUTPUT_MAP}")
print("Routes are colored by agency and sized by weekly trip frequency")
print("Stops use custom hollow circle with bus icon (cluster=", STOP_USE_MARKER_CLUSTER, ")")